In [19]:
# import zipfile, os
# from pathlib import Path

# src_dir = None
# for d in Path("/kaggle/input").rglob("task001.onnx"):
#     src_dir = d.parent; break
# print(f"src: {src_dir}")
# files = sorted(src_dir.glob("task*.onnx"))
# print(f"files: {len(files)}")

# dst = Path("/kaggle/working/submission.zip")
# with zipfile.ZipFile(dst, "w", zipfile.ZIP_DEFLATED) as zf:
#     for f in files:
#         zf.write(f, f.name)
# print(f"submission.zip: {dst.stat().st_size/1024:.1f}KB, {len(files)} files")

src: /kaggle/input/datasets/konbu17/neurogolf-2026-blend-source-v3-6-0
files: 400
submission.zip: 1044.0KB, 400 files


In [ ]:
# try:
#     import onnxruntime as _ort  # noqa: F401
# except ImportError:
#     subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime'])
#     import onnxruntime as _ort  # noqa: F40
    
# from IPython.display import display
# import sys
# import subprocess
# import json
# import math
# import os
# import shutil
# import zipfile
# from collections import Counter, defaultdict
# from pathlib import Path

# import numpy as np
# import onnx
# import onnxruntime as ort
# from onnx import TensorProto
# from onnx import helper as oh
# from onnx import numpy_helper as onh
# from onnx import version_converter

# from pathlib import Path
# import pandas as pd
# import json
# import os

# SEARCH_FILES = [
#     "arc_primitives.csv",
#     "arc_primitives.json"
# ]

# SEARCH_DIRS = [
#     "/kaggle/input",
#     "/kaggle/working"
# ]

# found_files = {}

# print("=" * 80)
# print("SEARCHING KAGGLE DIRECTORIES")
# print("=" * 80)

# for target in SEARCH_FILES:
#     found_files[target] = []

#     for root in SEARCH_DIRS:

#         root_path = Path(root)

#         if not root_path.exists():
#             continue

#         try:
#             for file in root_path.rglob(target):
#                 found_files[target].append(str(file))
#         except Exception as e:
#             print(f"Search error in {root}: {e}")

# for target, files in found_files.items():

#     print("\n" + "=" * 80)
#     print(target)
#     print("=" * 80)

#     if files:
#         for f in files:
#             print("FOUND:", f)
#     else:
#         print("NOT FOUND")

# print("\n")
# print("=" * 80)
# print("ALL CSV FILES")
# print("=" * 80)

# all_csv = []

# for root in SEARCH_DIRS:

#     root_path = Path(root)

#     if not root_path.exists():
#         continue

#     try:
#         for file in root_path.rglob("*.csv"):
#             all_csv.append(str(file))
#     except:
#         pass

# if all_csv:
#     for f in sorted(all_csv):
#         print(f)
# else:
#     print("No CSV files found")

# print("\n")
# print("=" * 80)
# print("ALL JSON FILES")
# print("=" * 80)

# all_json = []

# for root in SEARCH_DIRS:

#     root_path = Path(root)

#     if not root_path.exists():
#         continue

#     try:
#         for file in root_path.rglob("*.json"):
#             all_json.append(str(file))
#     except:
#         pass

# if all_json:
#     for f in sorted(all_json):
#         print(f)
# else:
#     print("No JSON files found")

# print("\n")
# print("=" * 80)
# print("CSV PREVIEW")
# print("=" * 80)

# csv_loaded = False

# for csv_path in found_files["arc_primitives.csv"]:

#     try:
#         df = pd.read_csv(csv_path)

#         print("\nLoaded:", csv_path)
#         print("Shape:", df.shape)

#         display(df.head(10))

#         csv_loaded = True
#         break

#     except Exception as e:
#         print("Failed:", csv_path)
#         print(e)

# if not csv_loaded:
#     print("arc_primitives.csv not found anywhere")

# print("\n")
# print("=" * 80)
# print("JSON PREVIEW")
# print("=" * 80)

# json_loaded = False

# for json_path in found_files["arc_primitives.json"]:

#     try:

#         with open(json_path, "r") as f:
#             data = json.load(f)

#         print("\nLoaded:", json_path)

#         if isinstance(data, dict):

#             keys = list(data.keys())[:3]

#             short = {}

#             for k in keys:
#                 short[k] = data[k]

#             print(json.dumps(short, indent=2))

#         elif isinstance(data, list):

#             print(json.dumps(data[:3], indent=2))

#         else:

#             print(type(data))

#         json_loaded = True
#         break

#     except Exception as e:

#         print("Failed:", json_path)
#         print(e)

# if not json_loaded:
#     print("arc_primitives.json not found anywhere")

# print("\n")
# print("=" * 80)
# print("DONE")
# print("=" * 80)

# NUM_TASKS = 400
# INPUT_ROOT = Path('/kaggle/input')
# WORKING = Path('/kaggle/working')
# WORKING.mkdir(parents=True, exist_ok=True)

# OUTPUT_ZIP = WORKING / 'submission.zip'

# # 6029 base bundle - 400 task ONNX, will be used verbatim for 397 of 400 tasks
# JSRDCHT_DATASET = 'neurogolf-6029-submission-bundle'

# # Competition data (task001.json .. task400.json) - used only for verifying hand-builds
# COMP_DIR = Path('/kaggle/input/competitions/neurogolf-2026')
# if not COMP_DIR.exists():
#     COMP_DIR = Path('/kaggle/input/neurogolf-2026')

# def find_onnx_root(slug):
#     for entry in INPUT_ROOT.rglob('task001.onnx'):
#         if slug in str(entry):
#             return entry.parent
#     raise FileNotFoundError('Could not locate task ONNX root for ' + slug)

# jsrdcht_dir = find_onnx_root(JSRDCHT_DATASET)
# print('jsrdcht 6029 bundle :', jsrdcht_dir)
# print('competition data    :', COMP_DIR)

# def encode_grid(grid):
#     arr = np.array(grid, dtype=np.int32)
#     h, w = arr.shape
#     t = np.zeros((1, 10, 30, 30), dtype=np.float32)
#     for r in range(h):
#         for c in range(w):
#             v = int(arr[r, c])
#             if 0 <= v < 10:
#                 t[0, v, r, c] = 1.0
#     return t

# def calculate_params(model):
#     n = 0
#     for init in model.graph.initializer:
#         n += int(math.prod(init.dims)) if init.dims else 1
#     for si in model.graph.sparse_initializer:
#         n += int(math.prod(si.values.dims)) if si.values.dims else 1
#     for node in model.graph.node:
#         if node.op_type != 'Constant':
#             continue
#         for attr in node.attribute:
#             if attr.name == 'value':
#                 n += int(math.prod(attr.t.dims)) if attr.t.dims else 1
#             elif attr.name == 'value_floats':
#                 n += len(attr.floats)
#             elif attr.name == 'value_ints':
#                 n += len(attr.ints)
#     return n

# def calculate_memory(model_path, examples, n_runs=3):
#     model = onnx.load(str(model_path))
#     onnx.checker.check_model(model, full_check=True)
#     graph = onnx.shape_inference.infer_shapes(model, strict_mode=True).graph

#     tensor_dtype = {}
#     tensor_static = {}
#     for vi in list(graph.input) + list(graph.value_info) + list(graph.output):
#         if not vi.type.HasField('tensor_type'):
#             continue
#         shape = vi.type.tensor_type.shape
#         if not shape.dim:
#             continue
#         dims = []
#         ok = True
#         for d in shape.dim:
#             if not d.HasField('dim_value') or d.dim_value <= 0:
#                 ok = False
#                 break
#             dims.append(d.dim_value)
#         if not ok:
#             continue
#         np_dt = onnx.helper.tensor_dtype_to_np_dtype(vi.type.tensor_type.elem_type)
#         tensor_dtype[vi.name] = np_dt
#         tensor_static[vi.name] = int(np.prod(dims)) * np.dtype(np_dt).itemsize

#     node_outputs = {n.name: list(n.output) for n in graph.node}

#     opts = ort.SessionOptions()
#     opts.enable_profiling = True
#     opts.log_severity_level = 3
#     sess = ort.InferenceSession(str(model_path), opts, providers=['CPUExecutionProvider'])
#     for p in examples[:n_runs]:
#         _ = sess.run(['output'], {'input': encode_grid(p['input'])})
#     trace_path = sess.end_profiling()
#     with open(trace_path) as f:
#         trace = json.load(f)
#     os.remove(trace_path)

#     tensor_runtime = {}
#     for event in trace:
#         if event.get('cat') != 'Node' or 'args' not in event:
#             continue
#         if 'output_type_shape' not in event['args']:
#             continue
#         node_name = event.get('name', '').replace('_kernel_time', '')
#         if node_name not in node_outputs:
#             continue
#         outs = node_outputs[node_name]
#         for i, shape_dict in enumerate(event['args']['output_type_shape']):
#             if i >= len(outs):
#                 continue
#             name = outs[i]
#             if name not in tensor_dtype:
#                 continue
#             itemsize = np.dtype(tensor_dtype[name]).itemsize
#             sz = itemsize * sum(int(math.prod(dims)) for dims in shape_dict.values())
#             tensor_runtime[name] = max(tensor_runtime.get(name, 0), sz)

#     total = 0
#     for name, static in tensor_static.items():
#         if name in ('input', 'output'):
#             continue
#         total += max(static, tensor_runtime.get(name, 0))
#     return total

# def cost_and_score(model_path, examples):
#     mem = calculate_memory(model_path, examples)
#     model = onnx.load(str(model_path))
#     params = calculate_params(model)
#     cost = mem + params
#     score = max(1.0, 25.0 - math.log(max(1.0, cost)))
#     return cost, score, mem, params

# def verify(model_path, examples):
#     sess = ort.InferenceSession(str(model_path), providers=['CPUExecutionProvider'])
#     n_pass = 0
#     for p in examples:
#         try:
#             out = sess.run(['output'], {'input': encode_grid(p['input'])})[0]
#             tgt = encode_grid(p['output']) > 0.0
#             if np.array_equal(out > 0.0, tgt):
#                 n_pass += 1
#         except Exception:
#             pass
#     return n_pass, len(examples)

# def load_examples(task_id):
#     p = COMP_DIR / f'task{task_id:03d}.json'
#     if p.exists():
#         return json.load(p.open())
#     for c in COMP_DIR.rglob(f'task{task_id:03d}.json'):
#         return json.load(c.open())
#     raise FileNotFoundError(f'task{task_id:03d}.json')
# import pandas as pd
# import json

# # 1. Display CSV shortly
# print("📊 CSV PREVIEW (Primitives Dataset):")
# try:
#     # Pointing to the new primitives CSV
#     df = pd.read_csv('/kaggle/working/arc_primitives.csv')
#     display(df.head(10)) 
# except Exception as e:
#     print(f"CSV not found: {e}")

# print("\n" + "="*50 + "\n")

# # 2. Display JSON shortly
# print("📄 JSON SNIPPET (First 3 entries):")
# try:
#     # Pointing to the new primitives JSON
#     with open('/kaggle/working/arc_primitives.json', 'r') as f:
#         data = json.load(f)
#         short_data = {k: data[k] for k in list(data.keys())[:3]}
#         print(json.dumps(short_data, indent=2))
# except Exception as e:
#     print(f"JSON not found: {e}")
# """Hand-build task277: label 8-pixel connected components 1 or 2 based on whether
# the component's cell count is unique among the 3 components in the grid.

# Rule (verified on 266/266 pairs): 8-connected components of color-8 pixels;
# component with unique cell-count gets color 2, others get color 1. Color 0
# passes through.

# Strategy without banned ops (Loop/Scan/NonZero/Unique/Compress):
#   1. mask = input channel 8; ch0 = input channel 0
#   2. Label propagation: init labels with arange(900) where mask=1 (BIG elsewhere),
#      iterate 5 times with MinPool (via -MaxPool of -x), re-mask after each pool.
#   3. Extract 3 component IDs by ReduceMin, then mask-out and repeat.
#   4. Count cells in each component via ReduceSum of equality masks.
#   5. Uniqueness: count_i is unique iff (count_i != count_j) AND (count_i != count_k).
#   6. Per-cell output value = 2 if its component count is unique else 1.
#   7. Build 10-channel output: ch0 = original ch0, ch1 = is_label_1, ch2 = is_label_2,
#      ch3..9 = 0.

# Always-3-components assumption holds across 266/266 task277 pairs.
# """
# from __future__ import annotations

# import json
# import math
# import sys
# from pathlib import Path

# import numpy as np
# import onnx
# from onnx import TensorProto, helper as oh, numpy_helper as onh


# ROOT = Path("/kaggle/working")
# OUT_PATH = ROOT / "task277.onnx"
# OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# BIG = 999.0  # > 30*30 = 900 max index; used as "unmarked" sentinel
# SHAPE_4D = [1, 1, 30, 30]
# PAD_TENSOR = np.array([0, 0, 1, 1, 0, 0, 1, 1], dtype=np.int64)  # 3x3 kernel pads by 1


# def fp(name, value, shape=None):
#     arr = np.array(value, dtype=np.float32)
#     if shape is not None:
#         arr = arr.reshape(shape)
#     return onh.from_array(arr, name=name)


# def i64(name, value):
#     return onh.from_array(np.array(value, dtype=np.int64), name=name)


# def build():
#     initializers = []

#     # Negated index tensor [1,1,30,30] with -(row*30+col) at each cell.
#     # We keep labels in negated form throughout label-propagation to save Neg ops.
#     neg_idx = -np.arange(900, dtype=np.float32).reshape(1, 1, 30, 30)
#     initializers.append(onh.from_array(neg_idx, name="neg_idx"))

#     initializers.append(fp("BIG", BIG))
#     initializers.append(fp("NEG_BIG", -BIG))
#     initializers.append(fp("ONE", 1.0))
#     initializers.append(fp("TWO", 2.0))
#     initializers.append(fp("ZERO", 0.0))
#     initializers.append(onh.from_array(PAD_TENSOR, name="pads_hw"))

#     # Slice indices for channel selection
#     initializers.append(i64("slice_starts_ch0", [0, 0, 0, 0]))
#     initializers.append(i64("slice_ends_ch0", [1, 1, 30, 30]))
#     initializers.append(i64("slice_starts_ch8", [0, 8, 0, 0]))
#     initializers.append(i64("slice_ends_ch8", [1, 9, 30, 30]))
#     initializers.append(i64("slice_axes", [0, 1, 2, 3]))
#     initializers.append(i64("reduce_axes_all", [0, 1, 2, 3]))

#     nodes = []

#     # ---- Step 1: extract mask (channel 8) and ch0 (channel 0) ----
#     nodes.append(oh.make_node("Slice", ["input", "slice_starts_ch0", "slice_ends_ch0", "slice_axes"], ["ch0"], name="slice_ch0"))
#     nodes.append(oh.make_node("Slice", ["input", "slice_starts_ch8", "slice_ends_ch8", "slice_axes"], ["mask"], name="slice_ch8"))
#     # mask_bool: re-used by all Where re-masks below
#     nodes.append(oh.make_node("Greater", ["mask", "ZERO"], ["mask_bool"], name="mask_bool"))

#     # Keep labels NEGATED throughout the loop to save per-iter Neg ops.
#     # nlabels_0 = Where(mask_bool, -idx, NEG_BIG). We store -idx as an initializer.
#     nodes.append(oh.make_node("Where", ["mask_bool", "neg_idx", "NEG_BIG"], ["nlabels_0"], name="nlabels_init"))

#     # ---- Step 2: 5 iterations of MaxPool 3x3 on NEGATED labels (equivalent to MinPool) ----
#     prev = "nlabels_0"
#     for it in range(5):
#         padded = f"pad_{it}"
#         nodes.append(oh.make_node("Pad", [prev, "pads_hw", "NEG_BIG"], [padded], mode="constant", name=f"pad_{it}"))
#         pooled = f"pooled_{it}"
#         nodes.append(oh.make_node("MaxPool", [padded], [pooled],
#                                   kernel_shape=[3, 3], strides=[1, 1], pads=[0, 0, 0, 0],
#                                   name=f"pool_{it}"))
#         nxt = f"nlabels_{it+1}"
#         nodes.append(oh.make_node("Where", ["mask_bool", pooled, "NEG_BIG"], [nxt], name=f"remask_{it}"))
#         prev = nxt
#     # Un-negate once at the end to get final labels for ReduceMin / Equal
#     nodes.append(oh.make_node("Neg", [prev], ["labels"], name="labels_final"))
#     labels = "labels"  # shape [1,1,30,30]

#     # ---- Step 3: extract 3 component IDs via successive ReduceMin + mask-out ----
#     # c1 = ReduceMin(labels, all)  -- shape [1,1,1,1]
#     nodes.append(oh.make_node("ReduceMin", [labels], ["c1"], axes=[0, 1, 2, 3], keepdims=1, name="rmin_c1"))
#     nodes.append(oh.make_node("Equal", [labels, "c1"], ["mask_c1_b"], name="eq_c1"))
#     nodes.append(oh.make_node("Cast", ["mask_c1_b"], ["mask_c1"], to=TensorProto.FLOAT, name="cast_c1"))

#     # labels_sup1 = where(mask_c1_b, BIG, labels)
#     nodes.append(oh.make_node("Where", ["mask_c1_b", "BIG", labels], ["labels_sup1"], name="suppress_c1"))

#     nodes.append(oh.make_node("ReduceMin", ["labels_sup1"], ["c2"], axes=[0, 1, 2, 3], keepdims=1, name="rmin_c2"))
#     nodes.append(oh.make_node("Equal", [labels, "c2"], ["mask_c2_b"], name="eq_c2"))
#     nodes.append(oh.make_node("Cast", ["mask_c2_b"], ["mask_c2"], to=TensorProto.FLOAT, name="cast_c2"))
#     nodes.append(oh.make_node("Where", ["mask_c2_b", "BIG", "labels_sup1"], ["labels_sup2"], name="suppress_c2"))

#     nodes.append(oh.make_node("ReduceMin", ["labels_sup2"], ["c3"], axes=[0, 1, 2, 3], keepdims=1, name="rmin_c3"))
#     nodes.append(oh.make_node("Equal", [labels, "c3"], ["mask_c3_b"], name="eq_c3"))
#     nodes.append(oh.make_node("Cast", ["mask_c3_b"], ["mask_c3"], to=TensorProto.FLOAT, name="cast_c3"))

#     # ---- Step 4: counts per component ----
#     nodes.append(oh.make_node("ReduceSum", ["mask_c1", "reduce_axes_all"], ["count1"], keepdims=1, name="rsum_c1"))
#     nodes.append(oh.make_node("ReduceSum", ["mask_c2", "reduce_axes_all"], ["count2"], keepdims=1, name="rsum_c2"))
#     nodes.append(oh.make_node("ReduceSum", ["mask_c3", "reduce_axes_all"], ["count3"], keepdims=1, name="rsum_c3"))

#     # ---- Step 5: uniqueness check ----
#     # ne_ij = NOT(count_i == count_j)
#     nodes.append(oh.make_node("Equal", ["count1", "count2"], ["eq12"], name="eq12"))
#     nodes.append(oh.make_node("Equal", ["count1", "count3"], ["eq13"], name="eq13"))
#     nodes.append(oh.make_node("Equal", ["count2", "count3"], ["eq23"], name="eq23"))
#     nodes.append(oh.make_node("Not", ["eq12"], ["ne12"], name="ne12"))
#     nodes.append(oh.make_node("Not", ["eq13"], ["ne13"], name="ne13"))
#     nodes.append(oh.make_node("Not", ["eq23"], ["ne23"], name="ne23"))
#     nodes.append(oh.make_node("And", ["ne12", "ne13"], ["unique1"], name="unique1"))
#     nodes.append(oh.make_node("And", ["ne12", "ne23"], ["unique2"], name="unique2"))
#     nodes.append(oh.make_node("And", ["ne13", "ne23"], ["unique3"], name="unique3"))

#     # ---- Step 6: per-component label value (2.0 if unique else 1.0) ----
#     nodes.append(oh.make_node("Where", ["unique1", "TWO", "ONE"], ["lv1"], name="lv1"))
#     nodes.append(oh.make_node("Where", ["unique2", "TWO", "ONE"], ["lv2"], name="lv2"))
#     nodes.append(oh.make_node("Where", ["unique3", "TWO", "ONE"], ["lv3"], name="lv3"))

#     # ---- Step 7: per-cell output value = sum of mask_ci * lv_i ----
#     nodes.append(oh.make_node("Mul", ["mask_c1", "lv1"], ["m1l"], name="m1l"))
#     nodes.append(oh.make_node("Mul", ["mask_c2", "lv2"], ["m2l"], name="m2l"))
#     nodes.append(oh.make_node("Mul", ["mask_c3", "lv3"], ["m3l"], name="m3l"))
#     nodes.append(oh.make_node("Sum", ["m1l", "m2l", "m3l"], ["val"], name="sum_vals"))  # shape [1,1,30,30]

#     # ---- Step 8: build 10 output channels ----
#     # is_1 = (val == 1).cast(fp32)
#     nodes.append(oh.make_node("Equal", ["val", "ONE"], ["is_1_b"], name="is1"))
#     nodes.append(oh.make_node("Cast", ["is_1_b"], ["is_1"], to=TensorProto.FLOAT, name="cast_is1"))
#     nodes.append(oh.make_node("Equal", ["val", "TWO"], ["is_2_b"], name="is2"))
#     nodes.append(oh.make_node("Cast", ["is_2_b"], ["is_2"], to=TensorProto.FLOAT, name="cast_is2"))

#     # zero_channel = ch0 * 0 (cheap way to get [1,1,30,30] of zeros)
#     nodes.append(oh.make_node("Mul", ["ch0", "ZERO"], ["zero_ch"], name="zero_ch"))

#     # Concat [ch0, is_1, is_2, zero, zero, zero, zero, zero, zero, zero] on axis 1
#     concat_inputs = ["ch0", "is_1", "is_2"] + ["zero_ch"] * 7
#     nodes.append(oh.make_node("Concat", concat_inputs, ["output"], axis=1, name="concat_out"))

#     input_vi = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, 10, 30, 30])
#     output_vi = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, 10, 30, 30])
#     graph = oh.make_graph(nodes=nodes, name="task277",
#                           inputs=[input_vi], outputs=[output_vi],
#                           initializer=initializers)
#     opset = oh.make_opsetid("", 14)
#     model = oh.make_model(graph, opset_imports=[opset], ir_version=8)
#     model.producer_name = ""
#     onnx.checker.check_model(model, full_check=True)
#     onnx.save(model, str(OUT_PATH))
#     return model

# _t277 = build()
# print(f'task277 built: {OUT_PATH.stat().st_size} bytes, {len(_t277.graph.node)} nodes')
# _t277_examples = load_examples(277)
# _t277_ex = _t277_examples['train'] + _t277_examples['test'] + _t277_examples.get('arc-gen', [])
# _t277_pass, _t277_total = verify(WORKING / 'task277.onnx', _t277_ex)
# _t277_cost, _t277_score, _, _ = cost_and_score(WORKING / 'task277.onnx', _t277_ex)
# print(f'task277: verify {_t277_pass}/{_t277_total}, cost {_t277_cost}, predicted score {_t277_score:.3f}')
# """task330 ONNX builder.

# Rule (verified 266/266): for each connected component of color-5 (8-conn),
# recolor cells to 2 if component has exactly 6 cells, else to 1.

# Strategy without banned ops:
#   1. Extract mask = ch_5. Label propagation via 5 iters of negated-MaxPool 3x3.
#   2. Compute per-cell component min-id label.
#   3. Build histogram via ScatterND with reduction='add' (opset 16+):
#      hist[label_at_cell] += 1 (for mask=1 cells).
#   4. Gather each cell's component count from hist.
#   5. color = 2 if count == 6 else 1, only for mask=1 cells.
#   6. Build output channels: ch_0 preserved, ch_1 = (mask AND count != 6), ch_2 = (mask AND count == 6).
# """
# from __future__ import annotations

# from pathlib import Path

# import numpy as np
# import onnx
# from onnx import TensorProto, helper as oh, numpy_helper as onh

# ROOT = Path("/kaggle/working")
# OUT_PATH = ROOT / "task330.onnx"

# BIG = 999.0
# PAD_TENSOR = np.array([0, 0, 1, 1, 0, 0, 1, 1], dtype=np.int64)


# def fp(name, value):
#     return onh.from_array(np.array(value, dtype=np.float32), name=name)


# def i64(name, value):
#     return onh.from_array(np.array(value, dtype=np.int64), name=name)


# def build():
#     inits = []

#     # Negated index tensor [1,1,30,30] fp16 to halve propagation memory.
#     neg_idx = -np.arange(900, dtype=np.float16).reshape(1, 1, 30, 30)
#     inits.append(onh.from_array(neg_idx, name="neg_idx"))

#     inits.append(onh.from_array(np.array(-BIG, dtype=np.float16), name="NEG_BIG"))
#     inits.append(fp("BIG", BIG))
#     inits.append(fp("ZERO", 0.0))
#     inits.append(fp("ONE", 1.0))
#     inits.append(fp("TWO", 2.0))
#     inits.append(fp("SIX", 6.0))

#     inits.append(onh.from_array(PAD_TENSOR, name="pads_hw"))

#     # Slice indices for channels
#     inits.append(i64("starts_ch0", [0, 0, 0, 0]))
#     inits.append(i64("ends_ch0", [1, 1, 30, 30]))
#     inits.append(i64("starts_ch5", [0, 5, 0, 0]))
#     inits.append(i64("ends_ch5", [1, 6, 30, 30]))
#     inits.append(i64("axes_all", [0, 1, 2, 3]))

#     # Hist size 1000 â€” accommodates the BIG=999 sentinel for mask=0 cells in an
#     # unused bucket, avoiding the clamp Where.
#     inits.append(i64("hist_shape", [1000]))
#     inits.append(i64("axes_unsqueeze_last", [4]))
#     # Static zero channel [1,1,30,30] for output-channel padding (saves 3600 bytes vs computed)
#     inits.append(onh.from_array(np.zeros((1, 1, 30, 30), dtype=np.float32), name="zero_ch"))

#     nodes = []

#     # ---- Step 1: extract channels ----
#     nodes.append(oh.make_node("Slice", ["input", "starts_ch0", "ends_ch0", "axes_all"], ["ch0"], name="slice_ch0"))
#     nodes.append(oh.make_node("Slice", ["input", "starts_ch5", "ends_ch5", "axes_all"], ["mask"], name="slice_ch5"))
#     nodes.append(oh.make_node("Greater", ["mask", "ZERO"], ["mask_bool"], name="mask_bool"))

#     # ---- Step 2: label propagation (4 iterations, verified max graph dist = 4) ----
#     # Use MaxPool's internal padding (auto -inf for max), avoiding explicit Pad nodes.
#     nodes.append(oh.make_node("Where", ["mask_bool", "neg_idx", "NEG_BIG"], ["nlabels_0"], name="nlabels_init"))
#     prev = "nlabels_0"
#     for it in range(4):
#         pooled = f"pooled_{it}"
#         nodes.append(oh.make_node("MaxPool", [prev], [pooled],
#                                   kernel_shape=[3, 3], strides=[1, 1], pads=[1, 1, 1, 1],
#                                   name=f"pool_{it}"))
#         nxt = f"nlabels_{it+1}"
#         nodes.append(oh.make_node("Where", ["mask_bool", pooled, "NEG_BIG"], [nxt], name=f"remask_{it}"))
#         prev = nxt
#     # Un-negate to get positive labels (in [0, 899] for mask=1 cells, BIG for mask=0)
#     nodes.append(oh.make_node("Neg", [prev], ["labels"], name="labels_final"))

#     # ---- Step 3: histogram via ScatterND-with-add ----
#     # Cast labels to int64 (ScatterND indices require int64). mask=0 cells have value
#     # BIG=999, which lands in the unused bucket of the 1000-sized histogram.
#     nodes.append(oh.make_node("Cast", ["labels"], ["labels_i64"], to=TensorProto.INT64, name="cast_labels"))
#     # Add trailing index dim: [1,1,30,30] -> [1,1,30,30,1] for ScatterND
#     nodes.append(oh.make_node("Unsqueeze", ["labels_i64", "axes_unsqueeze_last"], ["labels_idx"], name="unsq_idx"))
#     # Initialize hist [1000] = zeros
#     nodes.append(oh.make_node("ConstantOfShape", ["hist_shape"], ["hist_init"],
#                               value=onh.from_array(np.array([0.0], dtype=np.float32)),
#                               name="hist_init"))
#     # ScatterND with reduction='add': hist[label[i,j,k,l]] += mask[i,j,k,l]
#     nodes.append(oh.make_node("ScatterND", ["hist_init", "labels_idx", "mask"],
#                               ["hist"], reduction="add", name="scatter_hist"))

#     # ---- Step 4: gather per-cell count directly (no flatten) ----
#     nodes.append(oh.make_node("Gather", ["hist", "labels_i64"], ["count"], axis=0, name="gather_count"))

#     # ---- Step 5: build output channels directly from eq_6 + mask ----
#     nodes.append(oh.make_node("Equal", ["count", "SIX"], ["eq_6"], name="eq_6"))
#     # ch2_out = mask where size==6 else 0
#     nodes.append(oh.make_node("Where", ["eq_6", "mask", "ZERO"], ["ch2_out"], name="ch2_out"))
#     # ch1_out = mask where size!=6 else 0
#     nodes.append(oh.make_node("Where", ["eq_6", "ZERO", "mask"], ["ch1_out"], name="ch1_out"))
#     # ch_0_out = ch0 (cells of color 0 stay color 0)
#     # Concat 10 channels: [ch0, ch1, ch2, zero, zero, zero, zero, zero, zero, zero]
#     concat_inputs = ["ch0", "ch1_out", "ch2_out"] + ["zero_ch"] * 7
#     nodes.append(oh.make_node("Concat", concat_inputs, ["output"], axis=1, name="concat_out"))

#     input_vi = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, 10, 30, 30])
#     output_vi = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, 10, 30, 30])
#     graph = oh.make_graph(nodes=nodes, name="task330",
#                           inputs=[input_vi], outputs=[output_vi],
#                           initializer=inits)
#     # opset 17 (grader accepts; bundle's task277 uses 17)
#     opset = oh.make_opsetid("", 17)
#     model = oh.make_model(graph, opset_imports=[opset], ir_version=8)
#     model.producer_name = ""
#     onnx.checker.check_model(model, full_check=True)
#     OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
#     onnx.save(model, str(OUT_PATH))
#     return model

# _t330 = build()
# print(f'task330 built: {OUT_PATH.stat().st_size} bytes, {len(_t330.graph.node)} nodes')
# _t330_examples = load_examples(330)
# _t330_ex = _t330_examples['train'] + _t330_examples['test'] + _t330_examples.get('arc-gen', [])
# _t330_pass, _t330_total = verify(WORKING / 'task330.onnx', _t330_ex)
# _t330_cost, _t330_score, _, _ = cost_and_score(WORKING / 'task330.onnx', _t330_ex)
# print(f'task330: verify {_t330_pass}/{_t330_total}, cost {_t330_cost}, predicted score {_t330_score:.3f}')

# """task364 ONNX builder.

# Rule (verified 266/266): for each 4-connected component of color 3, recolor by topology:
#   - 3+ endpoints (cells with exactly 1 same-color 4-neighbor) -> color 2
#   - <=2 endpoints AND <=1 turns (cells with degree 2 where neighbors are NOT collinear) -> color 1
#   - <=2 endpoints AND >=2 turns -> color 6

# Strategy:
#   1. Extract mask = ch_3, plus other channels for output preservation.
#   2. Compute per-cell 4-neighbor presence: up, down, left, right (via Pad+Slice shifts).
#   3. deg_4 = up + down + left + right.
#      n_horiz = left + right.
#      n_vert = up + down.
#      is_endpoint = mask AND (deg_4 == 1).
#      is_turn = mask AND (n_horiz == 1) AND (n_vert == 1).
#   4. Label propagation via 8-conn MaxPool 3x3 (verified equivalent to 4-conn components for this task).
#      10 iterations covers max graph distance 10.
#   5. Two histograms via ScatterND-with-add:
#      hist_end[label] = sum of is_endpoint per component.
#      hist_turn[label] = sum of is_turn per component.
#   6. Per cell lookup: count_end = Gather(hist_end, label); count_turn = Gather(hist_turn, label).
#   7. Output channels:
#      ch_2 = mask AND (count_end >= 3)
#      ch_6 = mask AND (count_end <= 2) AND (count_turn >= 2)
#      ch_1 = mask AND (count_end <= 2) AND (count_turn <= 1)
# """
# from __future__ import annotations

# from pathlib import Path

# import numpy as np
# import onnx
# from onnx import TensorProto, helper as oh, numpy_helper as onh

# ROOT = Path("/kaggle/working")
# OUT_PATH = ROOT / "task364.onnx"


# def fp32(name, value):
#     return onh.from_array(np.array(value, dtype=np.float32), name=name)


# def i64(name, value):
#     return onh.from_array(np.array(value, dtype=np.int64), name=name)


# def build():
#     inits = []

#     # Negated index initializer in fp16 for label propagation
#     neg_idx = -np.arange(900, dtype=np.float16).reshape(1, 1, 30, 30)
#     inits.append(onh.from_array(neg_idx, name="neg_idx"))

#     # Constants
#     inits.append(onh.from_array(np.array(-999.0, dtype=np.float16), name="NEG_BIG"))
#     inits.append(fp32("ZERO", 0.0))
#     inits.append(fp32("ONE", 1.0))
#     inits.append(fp32("TWO", 2.0))
#     inits.append(fp32("THREE", 3.0))

#     # Slice indices for channels
#     inits.append(i64("starts_ch0", [0, 0, 0, 0]))
#     inits.append(i64("ends_ch0", [1, 1, 30, 30]))
#     inits.append(i64("starts_ch3", [0, 3, 0, 0]))
#     inits.append(i64("ends_ch3", [1, 4, 30, 30]))
#     inits.append(i64("axes_all", [0, 1, 2, 3]))

#     # Pad pads for shifting mask: 1 cell each side, 8 dims (N,C,H,W start, N,C,H,W end)
#     inits.append(i64("pads_top", [0, 0, 1, 0, 0, 0, 0, 0]))     # pad top -> shifts content down (or equivalently, "up_neighbor" needs pad top)
#     inits.append(i64("pads_bot", [0, 0, 0, 0, 0, 0, 1, 0]))     # pad bottom -> shifts content up
#     inits.append(i64("pads_left", [0, 0, 0, 1, 0, 0, 0, 0]))    # pad left
#     inits.append(i64("pads_right", [0, 0, 0, 0, 0, 0, 0, 1]))   # pad right

#     # Slice indices to crop back to [1,1,30,30] after padding
#     inits.append(i64("starts_crop_top", [0, 0, 0, 0]))
#     inits.append(i64("ends_crop_top", [1, 1, 30, 30]))      # take rows 0-29 (drops bottom row)
#     inits.append(i64("starts_crop_bot", [0, 0, 1, 0]))
#     inits.append(i64("ends_crop_bot", [1, 1, 31, 30]))      # take rows 1-30 (drops top row)
#     inits.append(i64("starts_crop_left", [0, 0, 0, 0]))
#     inits.append(i64("ends_crop_left", [1, 1, 30, 30]))     # take cols 0-29
#     inits.append(i64("starts_crop_right", [0, 0, 0, 1]))
#     inits.append(i64("ends_crop_right", [1, 1, 30, 31]))    # take cols 1-30

#     # For histogram
#     inits.append(i64("hist_shape", [1000]))
#     inits.append(i64("axes_unsqueeze_last", [4]))

#     # Static zero channel for output padding
#     inits.append(onh.from_array(np.zeros((1, 1, 30, 30), dtype=np.float32), name="zero_ch"))

#     nodes = []

#     # ---- Step 1: extract channels ----
#     nodes.append(oh.make_node("Slice", ["input", "starts_ch0", "ends_ch0", "axes_all"], ["ch0"], name="slice_ch0"))
#     nodes.append(oh.make_node("Slice", ["input", "starts_ch3", "ends_ch3", "axes_all"], ["mask"], name="slice_ch3"))
#     nodes.append(oh.make_node("Greater", ["mask", "ZERO"], ["mask_bool"], name="mask_bool"))

#     # ---- Step 2: compute neighbor presence (up, down, left, right) ----
#     # up_nbr[r,c] = mask[r-1,c]: shift content DOWN by 1 row = pad top with 0, take rows 0-29
#     nodes.append(oh.make_node("Pad", ["mask", "pads_top", "ZERO"], ["mask_padded_t"], mode="constant", name="pad_t"))
#     nodes.append(oh.make_node("Slice", ["mask_padded_t", "starts_crop_top", "ends_crop_top", "axes_all"], ["up_nbr"], name="slice_up"))

#     # down_nbr[r,c] = mask[r+1,c]: pad bottom, take rows 1-30
#     nodes.append(oh.make_node("Pad", ["mask", "pads_bot", "ZERO"], ["mask_padded_b"], mode="constant", name="pad_b"))
#     nodes.append(oh.make_node("Slice", ["mask_padded_b", "starts_crop_bot", "ends_crop_bot", "axes_all"], ["down_nbr"], name="slice_down"))

#     # left_nbr[r,c] = mask[r,c-1]: pad left, take cols 0-29
#     nodes.append(oh.make_node("Pad", ["mask", "pads_left", "ZERO"], ["mask_padded_l"], mode="constant", name="pad_l"))
#     nodes.append(oh.make_node("Slice", ["mask_padded_l", "starts_crop_left", "ends_crop_left", "axes_all"], ["left_nbr"], name="slice_left"))

#     # right_nbr[r,c] = mask[r,c+1]: pad right, take cols 1-30
#     nodes.append(oh.make_node("Pad", ["mask", "pads_right", "ZERO"], ["mask_padded_r"], mode="constant", name="pad_r"))
#     nodes.append(oh.make_node("Slice", ["mask_padded_r", "starts_crop_right", "ends_crop_right", "axes_all"], ["right_nbr"], name="slice_right"))

#     # n_vert = up + down; n_horiz = left + right
#     nodes.append(oh.make_node("Sum", ["up_nbr", "down_nbr"], ["n_vert"], name="sum_vert"))
#     nodes.append(oh.make_node("Sum", ["left_nbr", "right_nbr"], ["n_horiz"], name="sum_horiz"))
#     # deg_4 = n_vert + n_horiz
#     nodes.append(oh.make_node("Sum", ["n_vert", "n_horiz"], ["deg_4"], name="sum_deg"))

#     # is_endpoint = mask AND (deg_4 == 1)
#     nodes.append(oh.make_node("Equal", ["deg_4", "ONE"], ["deg_eq_1"], name="deg_eq_1"))
#     nodes.append(oh.make_node("And", ["mask_bool", "deg_eq_1"], ["is_end_bool"], name="is_end"))
#     nodes.append(oh.make_node("Cast", ["is_end_bool"], ["is_end"], to=TensorProto.FLOAT, name="cast_is_end"))

#     # is_turn = mask AND (n_horiz == 1) AND (n_vert == 1)
#     nodes.append(oh.make_node("Equal", ["n_horiz", "ONE"], ["nh_eq_1"], name="nh_eq_1"))
#     nodes.append(oh.make_node("Equal", ["n_vert", "ONE"], ["nv_eq_1"], name="nv_eq_1"))
#     nodes.append(oh.make_node("And", ["nh_eq_1", "nv_eq_1"], ["turn_pre"], name="turn_pre"))
#     nodes.append(oh.make_node("And", ["mask_bool", "turn_pre"], ["is_turn_bool"], name="is_turn"))
#     nodes.append(oh.make_node("Cast", ["is_turn_bool"], ["is_turn"], to=TensorProto.FLOAT, name="cast_is_turn"))

#     # ---- Step 3: label propagation (10 iter 3x3 MaxPool 8-conn) ----
#     nodes.append(oh.make_node("Where", ["mask_bool", "neg_idx", "NEG_BIG"], ["nlabels_0"], name="nlabels_init"))
#     prev = "nlabels_0"
#     for it in range(10):
#         pooled = f"pooled_{it}"
#         nodes.append(oh.make_node("MaxPool", [prev], [pooled],
#                                   kernel_shape=[3, 3], strides=[1, 1], pads=[1, 1, 1, 1],
#                                   name=f"pool_{it}"))
#         nxt = f"nlabels_{it+1}"
#         nodes.append(oh.make_node("Where", ["mask_bool", pooled, "NEG_BIG"], [nxt], name=f"remask_{it}"))
#         prev = nxt
#     nodes.append(oh.make_node("Neg", [prev], ["labels"], name="labels_final"))

#     # ---- Step 4: cast labels to int64 ----
#     nodes.append(oh.make_node("Cast", ["labels"], ["labels_i64"], to=TensorProto.INT64, name="cast_labels"))
#     nodes.append(oh.make_node("Unsqueeze", ["labels_i64", "axes_unsqueeze_last"], ["labels_idx"], name="unsq_idx"))

#     # ---- Step 5: two histograms ----
#     nodes.append(oh.make_node("ConstantOfShape", ["hist_shape"], ["hist_init"],
#                               value=onh.from_array(np.array([0.0], dtype=np.float32)),
#                               name="hist_init"))
#     nodes.append(oh.make_node("ScatterND", ["hist_init", "labels_idx", "is_end"],
#                               ["hist_end"], reduction="add", name="scatter_end"))
#     nodes.append(oh.make_node("ScatterND", ["hist_init", "labels_idx", "is_turn"],
#                               ["hist_turn"], reduction="add", name="scatter_turn"))

#     # ---- Step 6: Gather per-cell counts ----
#     nodes.append(oh.make_node("Gather", ["hist_end", "labels_i64"], ["count_end"], axis=0, name="gather_end"))
#     nodes.append(oh.make_node("Gather", ["hist_turn", "labels_i64"], ["count_turn"], axis=0, name="gather_turn"))

#     # ---- Step 7: build output channels ----
#     # ch_2 = mask AND (count_end >= 3) <=> (count_end > 2)
#     nodes.append(oh.make_node("Greater", ["count_end", "TWO"], ["end_ge3"], name="end_ge3"))
#     nodes.append(oh.make_node("And", ["mask_bool", "end_ge3"], ["is_2_bool"], name="is_2"))
#     nodes.append(oh.make_node("Cast", ["is_2_bool"], ["ch2_out"], to=TensorProto.FLOAT, name="cast_ch2"))

#     # ch_6 = mask AND NOT(count_end >= 3) AND (count_turn >= 2) <=> (count_end <= 2) AND (count_turn > 1)
#     nodes.append(oh.make_node("Greater", ["count_turn", "ONE"], ["turn_ge2"], name="turn_ge2"))
#     nodes.append(oh.make_node("Not", ["end_ge3"], ["end_le2"], name="end_le2"))
#     nodes.append(oh.make_node("And", ["mask_bool", "end_le2"], ["mask_no2"], name="mask_no2"))
#     nodes.append(oh.make_node("And", ["mask_no2", "turn_ge2"], ["is_6_bool"], name="is_6"))
#     nodes.append(oh.make_node("Cast", ["is_6_bool"], ["ch6_out"], to=TensorProto.FLOAT, name="cast_ch6"))

#     # ch_1 = mask AND NOT(is_2) AND NOT(is_6)
#     nodes.append(oh.make_node("Not", ["turn_ge2"], ["turn_le1"], name="turn_le1"))
#     nodes.append(oh.make_node("And", ["mask_no2", "turn_le1"], ["is_1_bool"], name="is_1"))
#     nodes.append(oh.make_node("Cast", ["is_1_bool"], ["ch1_out"], to=TensorProto.FLOAT, name="cast_ch1"))

#     # Concat 10 channels: [ch0, ch1, ch2, zero, zero, zero, ch6, zero, zero, zero]
#     concat_inputs = ["ch0", "ch1_out", "ch2_out", "zero_ch", "zero_ch", "zero_ch",
#                      "ch6_out", "zero_ch", "zero_ch", "zero_ch"]
#     nodes.append(oh.make_node("Concat", concat_inputs, ["output"], axis=1, name="concat_out"))

#     input_vi = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, 10, 30, 30])
#     output_vi = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, 10, 30, 30])
#     graph = oh.make_graph(nodes=nodes, name="task364",
#                           inputs=[input_vi], outputs=[output_vi],
#                           initializer=inits)
#     opset = oh.make_opsetid("", 17)
#     model = oh.make_model(graph, opset_imports=[opset], ir_version=8)
#     model.producer_name = ""
#     onnx.checker.check_model(model, full_check=True)
#     OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
#     onnx.save(model, str(OUT_PATH))
#     return model

# _t364 = build()
# print(f'task364 built: {OUT_PATH.stat().st_size} bytes, {len(_t364.graph.node)} nodes')
# _t364_examples = load_examples(364)
# _t364_ex = _t364_examples['train'] + _t364_examples['test'] + _t364_examples.get('arc-gen', [])
# _t364_pass, _t364_total = verify(WORKING / 'task364.onnx', _t364_ex)
# _t364_cost, _t364_score, _, _ = cost_and_score(WORKING / 'task364.onnx', _t364_ex)
# print(f'task364: verify {_t364_pass}/{_t364_total}, cost {_t364_cost}, predicted score {_t364_score:.3f}')
# OVERRIDES = {
#     277: WORKING / 'task277.onnx',
#     330: WORKING / 'task330.onnx',
#     364: WORKING / 'task364.onnx',
# }

# with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
#     for tid in range(1, NUM_TASKS + 1):
#         name = f'task{tid:03d}.onnx'
#         if tid in OVERRIDES:
#             data = OVERRIDES[tid].read_bytes()
#         else:
#             data = (jsrdcht_dir / name).read_bytes()
#         zf.writestr(name, data)

# with zipfile.ZipFile(OUTPUT_ZIP) as zf:
#     names = sorted(zf.namelist())
# print('submission built')
# print('  hand-built tasks :', sorted(OVERRIDES))
# print('  total ONNX       :', len(names))
# print('  zip size (bytes) :', OUTPUT_ZIP.stat().st_size)

# print(OUTPUT_ZIP)

In [22]:
!pip install -q onnxruntime onnx

In [23]:
import os, zipfile, re, io, csv, glob, time, traceback
import numpy as np
import onnxruntime as ort
import onnx
from pathlib import Path
from collections import Counter, defaultdict

ort.set_default_logger_severity(3)
print("Ready.")

Ready.


In [24]:
SOURCE_ZIPS = []
for zp in sorted(glob.glob('/kaggle/input/notebooks/**/submission.zip', recursive=True)):
    SOURCE_ZIPS.append((zp, Path(zp).parent.name))

print(f'Found {len(SOURCE_ZIPS)} sources:')
for zp, name in SOURCE_ZIPS:
    print(f'  {name}  ({os.path.getsize(zp)/1024/1024:.1f} MB)')

Found 0 sources:


In [25]:
def safe_load_model(raw_bytes, task_id=None):
    """
    Минимальный фикс: только приводим input/output к 'input'/'output'.
    Никаких инференс-тестов. Доверяем исходникам.
    Возвращает (success, model_bytes, cost)
    """
    try:
        model = onnx.load_model_from_string(raw_bytes)
        g = model.graph
        if not g.input or not g.output:
            return False, raw_bytes, float('inf')

        rename_map = {}
        old_in = g.input[0].name
        old_out = g.output[0].name
        if old_in != 'input':  rename_map[old_in] = 'input'
        if old_out != 'output': rename_map[old_out] = 'output'

        # Применяем переименование если нужно
        if rename_map:
            g.input[0].name = rename_map.get(old_in, old_in)
            g.output[0].name = rename_map.get(old_out, old_out)
            
            for node in g.node:
                node.input[:]  = [rename_map.get(n, n) for n in node.input]
                node.output[:] = [rename_map.get(n, n) for n in node.output]
                
            for vi in g.value_info:
                vi.name = rename_map.get(vi.name, vi.name)
            for init in g.initializer:
                init.name = rename_map.get(init.name, init.name)

        model.ir_version = 8
        fixed_bytes = model.SerializeToString()
        
        # Кост: кол-во параметров + размер файла
        params = sum(int(np.prod(i.dims)) for i in g.initializer if i.dims)
        cost = params + len(fixed_bytes)
        return True, fixed_bytes, cost

    except Exception:
        # Если вообще не парсится, пропускаем
        return False, raw_bytes, float('inf')

print('safe_load_model simplified.')


def _fix_subgraph(sg, rename_map):
    for node in sg.node:
        for i in range(len(node.input)):
            if node.input[i] in rename_map:
                node.input[i] = rename_map[node.input[i]]
        for i in range(len(node.output)):
            if node.output[i] in rename_map:
                node.output[i] = rename_map[node.output[i]]
        for attr in node.attribute:
            if attr.HasField('g'):
                _fix_subgraph(attr.g, rename_map)
            for s in attr.graphs:
                _fix_subgraph(s, rename_map)
    for vi in list(sg.input) + list(sg.output) + list(sg.value_info):
        if vi.name in rename_map:
            vi.name = rename_map[vi.name]

print('safe_load_model defined.')

safe_load_model simplified.
safe_load_model defined.


In [26]:
best_models = {}
model_costs = {}
source_tracker = {}
src_stats = defaultdict(lambda: {'ok':0,'fail':0,'won':0,'total':0})

t0 = time.time()

for zip_path, label in SOURCE_ZIPS:
    print(f'\n--- {label} ---')
    try:
        with zipfile.ZipFile(zip_path) as zf:
            for entry in zf.namelist():
                if not entry.endswith('.onnx'): continue
                m = re.search(r'task(\d{3})\.onnx', os.path.basename(entry))
                if not m: continue
                tid = f'task{m.group(1)}.onnx'
                src_stats[label]['total'] += 1
                
                raw = zf.read(entry)
                ok, data, cost = safe_load_model(raw, tid)
                
                if not ok:
                    src_stats[label]['fail'] += 1
                    continue
                
                src_stats[label]['ok'] += 1
                
                # Keep cheapest per task
                if tid not in best_models or cost < model_costs[tid]:
                    best_models[tid] = data
                    model_costs[tid] = cost
                    source_tracker[tid] = label
                    src_stats[label]['won'] += 1
    except Exception as e:
        print(f'  ERROR: {e}')
        traceback.print_exc()

    s = src_stats[label]
    print(f'  ok={s["ok"]} fail={s["fail"]} won={s["won"]}')

print(f'\n{"="*60}')
print(f'Blend done in {time.time()-t0:.1f}s: {len(best_models)}/400 tasks')
print(f'{"="*60}')

for src, cnt in Counter(source_tracker.values()).most_common():
    print(f'  {src}: {cnt}')


Blend done in 0.0s: 0/400 tasks


In [27]:
def make_fallback():
    """Minimal valid ONNX: Conv 1x1 identity. Always works."""
    w = np.eye(10, dtype=np.float32).reshape(10,10,1,1)
    b = np.zeros(10, dtype=np.float32)
    node = onnx.helper.make_node('Conv', ['input','w','b'], ['output'],
                                 kernel_shape=[1,1], pads=[0,0,0,0])
    g = onnx.helper.make_graph([node], 'fb',
        [onnx.helper.make_tensor_value_info('input', onnx.TensorProto.FLOAT, [1,10,30,30])],
        [onnx.helper.make_tensor_value_info('output', onnx.TensorProto.FLOAT, [1,10,30,30])],
        [onnx.helper.make_tensor('w', onnx.TensorProto.FLOAT, [10,10,1,1], w.flatten()),
         onnx.helper.make_tensor('b', onnx.TensorProto.FLOAT, [10], b)])
    m = onnx.helper.make_model(g, opset_imports=[onnx.helper.make_opsetid('',12)])
    m.ir_version = 8
    return m.SerializeToString()

fb = make_fallback()
for i in range(400):
    tid = f'task{i:03d}.onnx'
    if tid not in best_models:
        best_models[tid] = fb
        model_costs[tid] = 10**8
        source_tracker[tid] = 'fallback'

print(f'Total models: {len(best_models)}/400')

Total models: 400/400


In [28]:
# Write submission.zip
buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
    for n in sorted(best_models):
        zf.writestr(n, best_models[n])

with open('submission.zip','wb') as f:
    f.write(buf.getvalue())

# Write submission.csv
with open('submission.csv','w',newline='') as f:
    w = csv.writer(f)
    w.writerow(['task_id','total_cost'])
    for n in sorted(best_models):
        w.writerow([n.replace('.onnx',''), model_costs[n]])

sz = buf.tell()
print(f'\n{"="*60}')
print(f'submission.zip: {sz/1024:.1f} KB, {len(best_models)} models')
print(f'Press Submit!')
print(f'{"="*60}')


submission.zip: 99.2 KB, 400 models
Press Submit!
